# 📝 RAG 검색 평가 과제 — 지표 네 개부터 k 고르기까지

> 교안에서 배운 **네 눈금**을 직접 만들고, 그 눈금으로 실제 검색기를 재어 **쓸 `k` 를 근거를 가지고** 고르는 데까지 갑니다. 1~4번은 개념 하나씩, 5~7번은 그 넷을 조합합니다.

## 풀이 방법
1. 문제마다 필요한 **준비 셀**(제공 코드)이 바로 위에 있습니다. 위에서부터 순서대로 실행하세요.
2. 각 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요.

- **이 과제는 모델을 한 번도 부르지 않습니다** — `OPENAI_API_KEY` 도 필요 없습니다. 임베딩과 검색만 쓰므로 값이 **언제나 같고**, 그래서 채점이 숫자를 정확히 봅니다.
- **1~4번**은 데이터도 색인도 필요 없는 **순수 리스트 연산**입니다.
- **5~7번**은 여러분이 1~4번에서 만든 **그 함수들을 그대로** 씁니다 — 다시 만들지 마세요. (1~4번을 건너뛰면 5번부터 `NameError` 가 납니다.)

화이팅!

## 1. 검색 평가 지표 — Hit@K

**배경**: 검색이 쓸 만한지 알려면 **재야** 합니다. 질문마다 검색 결과 상위 K개와 미리 정해 둔 정답 목록을 견주어 점수를 매기는 것이 검색 평가입니다. 1~4번에서 그 눈금 네 개를 하나씩 손으로 만듭니다.

**네 함수의 인자는 모두 같습니다.** `ranked` 는 검색 결과 조각 id 를 **1위부터 순서대로** 담은 리스트, `gold` 는 그 질문의 **정답 조각 id 리스트**, `k` 는 상위 몇 개까지 볼지입니다. 보는 값은 같고 **무엇을 세느냐만** 다릅니다.

**요구사항**:
- 함수 **`hit_at_k(ranked, gold, k) -> float`** 를 정의하세요(모델도 데이터도 쓰지 않는 순수 계산입니다).
- Hit@K 는 **맞혔나 못 맞혔나만** 봅니다. 상위 `k` 개 안에 정답이 **하나라도** 있으면 `1.0`, 하나도 없으면 `0.0` 을 돌려줍니다.
- 돌려주는 값은 **실수(float)** 입니다 — `True`/`False` 나 정수 `1`/`0` 이 아닙니다. 네 지표는 결국 **여러 질문에 걸쳐 평균을 내는 값**이라, 나머지 셋(소수가 나옵니다)과 자료형을 맞춰 둡니다.

**예시**

```
hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 1.0
hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 1)   -> 0.0   (1위만 보면 정답이 없다)
```

<details><summary>힌트</summary>

```text
접근방법:
- 상위 k개만 잘라 정답 목록과 겹치는 것이 있는지 보면 된다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 하나라도 정답 목록에 들어 있으면 1.0, 아니면 0.0 을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) == 1.0
assert hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 1) == 0.0   # k 밖의 정답은 세지 않는다
assert hit_at_k(['a', 'b'], ['z'], 2) == 0.0                  # 정답이 하나도 없을 때
assert hit_at_k(['a', 'b', 'c'], ['c', 'b'], 2) == 1.0        # 정답이 여럿이어도 하나만 맞으면 된다
assert type(hit_at_k(['a'], ['a'], 1)) is float, '1.0 또는 0.0(실수)을 돌려주세요'
print('✅ 통과!')

## 2. 검색 평가 지표 — Precision@K

**요구사항**: 함수 **`precision_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- Precision@K 는 **내가 꺼내 온 것 중 몇 개가 정답이었나**를 봅니다.
- 상위 `k` 개 안에 든 정답의 개수를 세어 **`k` 로 나눈 값**을 돌려줍니다.
- 나누는 수는 검색 결과의 길이가 아니라 언제나 **`k`** 입니다. 결과가 `k` 개보다 적게 와도 `k` 로 나눕니다.

**예시**

```
precision_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.333...   (3개 중 1개가 정답)
precision_at_k(['a', 'b', 'c'], ['b', 'c'], 3)        -> 0.666...   (3개 중 2개가 정답)
```

<details><summary>힌트</summary>

```text
접근방법:
- 상위 k개 중 정답 목록에 든 것의 개수를 세어 k로 나눈다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 정답 목록에 있는 것의 개수를 센다.
3. 그 개수를 k로 나눠 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 나눗셈 결과라 부동소수 오차가 있을 수 있어 정확일치 대신 아주 작은 차이로 비교한다
assert abs(precision_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) - 1 / 3) < 1e-9
assert abs(precision_at_k(['a', 'b', 'c'], ['b', 'c'], 3) - 2 / 3) < 1e-9
assert precision_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0        # 정답이 하나도 없을 때
assert abs(precision_at_k(['b'], ['b'], 3) - 1 / 3) < 1e-9, '결과가 k개보다 적어도 k 로 나눠야 합니다'
print('✅ 통과!')

## 3. 검색 평가 지표 — Recall@K

**요구사항**: 함수 **`recall_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- Recall@K 는 **찾았어야 할 정답 중 몇 개를 건졌나**를 봅니다.
- 상위 `k` 개 안에 든 정답의 개수를 세어 **그 질문의 전체 정답 개수(`gold` 의 길이)로 나눈 값**을 돌려줍니다.
- 세는 방법은 2번과 같고 **나누는 수만 다릅니다.**

**예시**

```
recall_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.333...   (정답 3개 중 1개를 건졌다)
recall_at_k(['a', 'b', 'c'], ['b'], 3)             -> 1.0        (정답 1개를 다 건졌다)
```

<details><summary>힌트</summary>

```text
접근방법:
- 2번과 세는 방법은 같고 나누는 수만 다르다 — 이번에는 전체 정답 개수로 나눈다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 정답 목록에 있는 것의 개수를 센다.
3. 그 개수를 전체 정답 개수로 나눠 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(recall_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) - 1 / 3) < 1e-9
assert recall_at_k(['a', 'b', 'c'], ['b', 'x'], 3) == 0.5
assert recall_at_k(['a', 'b', 'c'], ['b'], 3) == 1.0
assert recall_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0           # 정답이 하나도 없을 때
# 정답이 k 보다 많으면 아무리 잘 검색해도 1.0 이 될 수 없다
assert abs(recall_at_k(['a', 'b', 'c'], ['a', 'b', 'c', 'd'], 3) - 0.75) < 1e-9
print('✅ 통과!')

## 4. 검색 평가 지표 — MRR@K

**요구사항**: 함수 **`mrr_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- MRR@K 는 **정답이 얼마나 위쪽에 있었나**를 봅니다. 상위 `k` 개를 1위부터 보다가 **처음 만난 정답의 순위로 1 을 나눈 값**을 돌려줍니다(1위면 `1.0`, 2위면 `0.5`, 4위면 `0.25`).
- 정답을 두 개 이상 만나도 **처음 만난 것 하나만** 씁니다.
- 상위 `k` 개 안에 정답이 하나도 없으면 `0.0` 을 돌려줍니다. `k` 밖의 정답은 세지 않습니다.

**예시**

```
mrr_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.5   (첫 정답이 2위)
mrr_at_k(['b', 'a', 'c'], ['b', 'x'], 3)        -> 1.0   (첫 정답이 1위)
```

<details><summary>힌트</summary>

```text
접근방법:
- 앞에서부터 순서대로 보다가 처음 만난 정답의 순위를 쓰면 된다.

세부구현:
1. 상위 k개를 순위 번호와 함께 앞에서부터 돈다(첫 번째가 1위).
2. 정답 목록에 있는 것을 처음 만나면 1을 그 순위로 나눠 반환한다.
3. 끝까지 못 만나면 0.0 을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert mrr_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) == 0.5
assert mrr_at_k(['b', 'a', 'c'], ['b', 'x'], 3) == 1.0        # 1위에 있으면 1.0
assert mrr_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0             # 정답이 하나도 없을 때
assert mrr_at_k(['a', 'b', 'c'], ['c'], 2) == 0.0             # k 밖의 정답은 세지 않는다
assert mrr_at_k(['a', 'b', 'c'], ['b', 'c'], 3) == 0.5        # 정답이 여럿이어도 처음 하나만
print('✅ 통과!')

## 5. 검색 품질 재기 — 평가셋 전체를 네 지표로
**배경**: 여기까지는 손으로 만든 짧은 리스트로 지표를 확인했습니다. 이제 **진짜 검색기**에 붙입니다. 아래 평가셋에는 질문 25개마다 **정답 조각 id** 가 붙어 있습니다. 문항마다 검색을 하고 1~4번에서 만든 네 지표를 계산해 평균을 내면 이 검색기의 **성적표**가 됩니다.

> 이 문제의 코퍼스는 교안이 쓴 안내서가 아니라 **개인정보 질의응답 모음집**입니다. 아래 **제공 셀 세 개**(평가 코퍼스 살펴보기 → 청킹 → 색인)를 위에서부터 실행한 뒤 푸세요. 검색은 **`search_ids(질문, k)`** 로 하고, 조각 id 를 1위부터 순서대로 돌려줍니다. 지표는 1~4번에서 만든 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 그대로 씁니다(다시 만들지 마세요 — 인자는 그때와 같은 `(ranked, gold, k)` 입니다).

**요구사항**: `K = 3` 으로 평가셋 전체를 재서 세 가지를 만드세요.

- 문항마다 상위 3개 조각 id 를 얻습니다. 검색은 문항당 **한 번만** 하고, 그 결과 하나로 네 지표를 모두 계산합니다.
- 정답 라벨 `gold_chunks` 는 `'|'` 로 이어져 있으니 **나눠서 리스트로** 만들어 넘깁니다.
- 문항별 결과를 DataFrame **`eval_results`** 로 만드세요. 열은 **`query_id`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**여야 합니다. 한 행이 한 문항이므로 행 수는 평가 문항 수와 같습니다.
- 네 지표 각각의 평균을 딕셔너리 **`eval_summary`** 에 담으세요. 열쇠는 **`'Hit'`, `'P'`, `'R'`, `'MRR'`** 네 개입니다.
- `Hit` 이 `0.0` 인 문항의 `query_id` 를 리스트 **`missed_qna`** 에 담으세요(평가셋에 나온 순서 그대로).

**예시**: 제대로 재면 평균은 **Hit 약 0.96 · P 약 0.53 · R 약 0.72 · MRR 약 0.90** 근처가 나오고, `missed_qna` 에는 **문항 하나**만 남습니다 — 어느 질문이 걸렸는지 직접 열어 보세요. 환경에 따라 값이 조금 다를 수 있어 채점은 **넉넉한 범위**로 봅니다(값을 정확히 맞힐 필요는 없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 문항을 한 번씩 돌면서 검색을 한 번만 하고, 그 결과 목록으로 네 지표를 모두 계산한다.

세부구현:
1. 빈 목록을 만들어 두고 평가셋을 한 행씩 돈다.
   1-1. 질문으로 상위 3개 조각 id 를 얻는다.
   1-2. 정답 라벨을 구분자로 나눠 목록으로 만든다.
   1-3. 네 지표를 계산해 딕셔너리 하나로 담아 목록에 넣는다(적은 순서가 곧 열 순서다).
2. 목록으로 DataFrame 을 만든다.
3. 네 열의 평균을 딕셔너리로 만든다.
4. Hit 이 0 인 행만 골라 그 문항 번호를 목록으로 뽑는다.
```

</details>

In [ ]:
# [제공 코드] 평가 코퍼스와 평가셋을 읽고 눈으로 확인합니다 — 이 셀은 실행만 하세요.
import pandas as pd

# 문서 모음 — 한 행이 문서 하나이고, 검색 대상 글은 '본문' 열에 있습니다.
qna_df = pd.read_csv('data/qna_docs.csv')
# 평가셋 — 질문(query)마다 정답 조각 id(gold_chunks)가 '|' 로 이어져 붙어 있습니다.
evalset = pd.read_csv('data/qna_eval_chunk.csv')

print('문서', len(qna_df), '건 / 평가 문항', len(evalset), '건')
display(qna_df[['id', '분야', '질문']].head(3))
display(evalset[['query_id', 'query', 'gold_chunks', '유형']].head(3))

In [ ]:
# 청킹 — 19일차에서 배운 그 스플리터입니다(문단 -> 줄 -> 문장 -> 낱말 순으로 큰 경계부터 존중합니다).
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=400 : 안내서 한 쪽이 길어 반드시 잘립니다. chunk_overlap=80 은 경계에서 문장이 반 토막 나는 것을 막아 줍니다.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
print('스플리터 준비 완료')

In [ ]:
# [제공 코드] 문서를 조각으로 나눠 색인하고 검색 함수를 만듭니다 — 이 셀은 실행만 하세요.
#  (임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다.)
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# 조각 id 는 '문서id-순번' 규칙입니다. 평가셋의 정답 라벨이 이 규칙으로 붙어 있으므로
#  청킹 기준이나 id 규칙을 바꾸면 정답과 어긋나 점수가 전부 달라집니다.
qna_chunks, qna_ids = [], []
for doc_id, text in zip(qna_df['id'], qna_df['본문']):
    for i, part in enumerate(splitter.split_text(text)):
        qna_chunks.append(Document(page_content=part))
        qna_ids.append(f'{doc_id}-{i}')

qna_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 조각이 중복되지 않습니다.
#  ids 로 넘긴 값이 곧 그 조각의 열쇠이고, 검색 결과의 Document.id 로 그대로 돌아옵니다.
qna_store = Chroma.from_documents(qna_chunks, qna_embeddings, collection_name='qna_eval',
                                  ids=qna_ids)


def search_ids(query, k):
    """질문과 의미가 가까운 조각 id 를 1위부터 k개까지 순서대로 돌려준다."""
    # 평가에서는 k 를 바꿔 가며 재야 해서, k 를 인자로 받는 검색을 그대로 씁니다.
    return [d.id for d in qna_store.similarity_search(query, k=k)]


print('조각', len(qna_chunks), '개 색인 완료')
print('첫 문항 검색 결과:', search_ids(evalset['query'].iloc[0], 3))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 값을 손으로 적어 넣으면 통과하지 못하도록, 채점이 검색을 다시 돌려 정답을 스스로 계산한다
# (임베딩·검색은 결정적이라 같은 질문·같은 k 면 결과가 항상 같다)
assert list(eval_results.columns) == ['query_id', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert len(eval_results) == len(evalset), '모든 문항을 재야 합니다'

for query_id, query, gold_text in zip(evalset['query_id'], evalset['query'], evalset['gold_chunks']):
    ranked = search_ids(query, 3)
    gold = gold_text.split('|')
    mine = eval_results[eval_results['query_id'] == query_id].iloc[0]
    assert mine['Hit'] == hit_at_k(ranked, gold, 3), f'{query_id} Hit'
    assert abs(mine['P'] - precision_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} P'
    assert abs(mine['R'] - recall_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} R'
    assert abs(mine['MRR'] - mrr_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} MRR'

# 평균은 값을 정확히 맞히는 대신 '이 언저리인지'만 본다
#  (임베딩 모델 버전이 달라 한두 문항이 뒤집혀도 옳은 풀이는 통과하고,
#   분모를 잘못 쓴 계산은 다른 지표 값으로 넘어가 버리므로 범위 밖으로 떨어진다)
for metric, low, high in [('Hit', 0.85, 1.00), ('P', 0.45, 0.65),
                          ('R', 0.65, 0.85), ('MRR', 0.85, 1.00)]:
    assert abs(eval_summary[metric] - eval_results[metric].mean()) < 1e-9, f'eval_summary[{metric}] 가 다릅니다'
    assert low < eval_summary[metric] <= high, f'{metric} 평균이 지문에 적힌 언저리를 벗어났습니다'

assert missed_qna == eval_results.loc[eval_results['Hit'] == 0.0, 'query_id'].tolist()
assert len(missed_qna) == 1, '상위 3개 안에 정답이 하나도 없는 문항은 한 건입니다'
print('✅ 통과!')

## 6. K 를 바꿔 가며 비교 — 넓게 볼수록 좋을까
**배경**: 5번에서는 K 를 3 으로 고정했습니다. K 는 **상위 몇 개를 채택할지 정하는 파라미터**입니다. 어느 값이 나은지는 **재 봐야** 압니다.

**요구사항**: `K = 1, 3, 5, 10` 네 가지로 각각 평가셋 전체를 재서 DataFrame **`k_table`** 을 만드세요.

- 열은 **`K`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**입니다. 한 행이 K 하나이고, 행 순서는 **1, 3, 5, 10** 입니다.
- 각 칸은 그 K 로 잰 **전 문항 평균**입니다(5번과 같은 방법, K 만 바꿉니다). 지표는 5번과 마찬가지로 1~4번의 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 씁니다.
- 검색은 문항마다 **가장 큰 K(=10)로 한 번만** 하고, 그 목록 하나로 네 K 를 모두 계산하세요. 지표 함수가 알아서 앞에서 `k` 개만 보므로 목록을 다시 자를 필요가 없습니다.

**예시**: 위의 두 행은 대략 이렇게 나옵니다(소수 셋째 자리까지 — 환경에 따라 조금 다를 수 있어 채점은 **넉넉한 범위와 방향**으로 봅니다).

```
 K    Hit      P      R    MRR
 1  0.840  0.840  0.435  0.840
 3  0.960  0.533  0.717  0.900
 5    ...    ...    ...    ...
10    ...    ...    ...    ...
```

표를 다 채우면 **K 를 키울 때 오르는 지표와 내려가는 지표가 갈립니다.** 그 모습을 확인하고 아래 서술 답안을 채우세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 상위 10개를 한 번 받아 두고, 같은 목록에 k 만 바꿔 넣어 네 번 잰다.

세부구현:
1. 평가셋을 한 번 돌며 (상위 10개 조각 id, 정답 목록) 짝을 모아 둔다.
2. K 후보 네 개를 차례로 돈다.
   2-1. 모아 둔 짝마다 네 지표를 계산해 문항 수로 나눠 평균을 낸다.
   2-2. K 와 네 평균을 딕셔너리 하나로 담아 목록에 넣는다.
3. 목록으로 DataFrame 을 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert k_table['K'].tolist() == [1, 3, 5, 10], 'K 는 1, 3, 5, 10 순서여야 합니다'

# 채점도 같은 방식으로 다시 재서 대조한다 — 표에 값을 적어 넣는 것으로는 통과하지 못한다
pairs = [(search_ids(query, 10), gold_text.split('|'))
         for query, gold_text in zip(evalset['query'], evalset['gold_chunks'])]
for i, k in enumerate([1, 3, 5, 10]):
    for metric, metric_fn in [('Hit', hit_at_k), ('P', precision_at_k),
                              ('R', recall_at_k), ('MRR', mrr_at_k)]:
        want = sum(metric_fn(r, g, k) for r, g in pairs) / len(pairs)
        assert abs(k_table.loc[i, metric] - want) < 1e-9, f'K={k} 의 {metric} 값이 다릅니다'

# 지문에 적어 둔 K=1 · K=3 행이 그 언저리인지 본다(정확한 값이 아니라 범위로)
assert 0.80 < k_table.loc[0, 'Hit'] <= 1.00, 'K=1 의 Hit 가 지문의 언저리를 벗어났습니다'
assert 0.35 < k_table.loc[0, 'R'] < 0.55, 'K=1 의 Recall 이 지문의 언저리를 벗어났습니다'
assert 0.45 < k_table.loc[1, 'P'] < 0.65, 'K=3 의 Precision 이 지문의 언저리를 벗어났습니다'

# K 를 키우면 Recall 은 오르고 Precision 은 떨어진다 -- 맞바꿈이 표에 그대로 보인다
assert k_table.loc[3, 'R'] > k_table.loc[0, 'R'], 'K 가 커지면 Recall 은 올라야 합니다'
assert k_table.loc[3, 'P'] < k_table.loc[0, 'P'], 'K 가 커지면 Precision 은 내려야 합니다'
print('✅ 통과!')

**서술 답안** — 표를 보고 아래에 적으세요.

*(여기에 이 검색기의 K 를 얼마로 정할지, 표의 어떤 값을 근거로 그렇게 정했는지 서술하세요)*

## 7. 검색기를 재서 `k` 를 정하기
**배경**: 마지막 문제는 **다른 검색기**를 재서 실제 서비스에 넣을 `k` 를 **근거를 가지고** 고릅니다. 서점 고객센터 FAQ 22건을 색인하고, 손으로 만든 평가셋 14문항으로 잽니다.

지금까지는 `k=2` 나 `k=3` 을 그냥 썼습니다. 이번에는 **왜 그 값인지**를 수치로 말할 수 있게 만듭니다.

제공 셀이 색인(`bs_store`)과 검색 함수(`bs_search_ids`), 평가셋(`bs_eval`)을 만들어 둡니다. 지표는 1~4번에서 만든 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 그대로 씁니다 — 3단계에서 `hit_at_k` 로 못 찾은 문항을 가려낼 때도 같은 함수를 씁니다.

> 검색 함수와 표 이름이 `bs_` 로 시작합니다 — 위 5·6번이 만든 `search_ids`·`k_table`(질의응답 코퍼스)을 덮어쓰면 **에러 없이 엉뚱한 코퍼스를 재게** 되기 때문입니다.

In [ ]:
# [제공 코드] 서점 FAQ 색인과 평가셋 — 이 셀은 실행만 하세요(임베딩에 잠시 걸립니다).
#  FAQ 한 행이 조각 하나입니다(한 건이 짧아 자를 것이 없습니다) -> 조각 id 는 FAQ 의 id 그대로입니다.
#  검색 함수 이름이 bs_ 로 시작합니다 -- 위 문제들의 search_ids(질의응답 코퍼스)를 덮어쓰지 않기 위해서입니다.
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

bs_faq = pd.read_csv('data/bookstore_faq.csv')
bs_eval = pd.read_csv('data/bookstore_eval.csv')

bs_documents = [Document(page_content=f'{r.title}\n{r.text}') for r in bs_faq.itertuples()]
bs_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
bs_store = Chroma.from_documents(bs_documents, bs_embeddings,
                                 collection_name='bookstore_eval', ids=list(bs_faq['id']))


def bs_search_ids(query, k):
    """질문과 가장 가까운 서점 FAQ k개의 id 를 순위 순서로 돌려준다."""
    return [d.id for d in bs_store.as_retriever(search_kwargs={'k': k}).invoke(query)]


print('FAQ', len(bs_faq), '건 색인 / 평가셋', len(bs_eval), '문항')
display(bs_eval.head(3))

### 1단계 — K 별로 재서 표 만들기

`K` 를 **1·2·3·5** 로 바꿔 가며 평가셋 **전체 평균**을 재고, 그 결과를 DataFrame **`bs_k_table`** 에 담으세요.

- 정답 라벨은 `bs_eval` 의 **`gold_chunks`** 열에 `'|'` 로 이어져 있습니다 — 나눠서 리스트로 쓰세요.
- 검색은 제공된 **`bs_search_ids(query, k)`** 를 쓰세요.
- `bs_k_table` 의 열은 **`['K', 'Hit', 'P', 'R', 'MRR']`** 이고 행은 K 오름차순 **4행**입니다.
- 각 지표 값은 그 K 로 잰 **14문항 평균**입니다.

**예시**: `bs_k_table` 의 첫 행은 `K=1` 이고, 그때 `Hit` 은 0.9 보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복은 K, 안쪽 반복은 문항이다. 문항마다 한 번만 검색해 네 지표를 모두 뽑는다.

세부구현:
1. K 값 네 개를 순서대로 돈다.
2. 그 안에서 평가셋의 행을 돌며
   2-1. 질문으로 상위 K개 id 를 찾고
   2-2. 정답 문자열을 구분자로 나눠 리스트로 만들고
   2-3. 네 지표를 각각 계산해 모아 둔다.
3. 네 지표의 평균을 그 K 의 한 행으로 담는다.
4. 행들을 DataFrame 으로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(bs_k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 맞춰 주세요'
assert list(bs_k_table['K']) == [1, 2, 3, 5]
# 재현율은 K 가 커질수록 줄어들 수 없고, 정밀도는 커질수록 늘어날 수 없다(구조적 성질)
assert list(bs_k_table['R']) == sorted(bs_k_table['R'])
assert list(bs_k_table['P']) == sorted(bs_k_table['P'], reverse=True)
# 실측값과 대조 - 검색은 결정적이라 같은 색인에서 같은 값이 나온다
assert abs(bs_k_table.loc[0, 'Hit'] - 0.857) < 0.02, 'K=1 의 Hit 이 실측과 다릅니다'
assert abs(bs_k_table.loc[0, 'R'] - 0.643) < 0.02, 'K=1 의 Recall 이 실측과 다릅니다'
assert abs(bs_k_table.loc[1, 'R'] - 0.929) < 0.02, 'K=2 의 Recall 이 실측과 다릅니다'
assert abs(bs_k_table.loc[3, 'P'] - 0.271) < 0.02, 'K=5 의 Precision 이 실측과 다릅니다'
print('✅ 통과!')

### 2단계 — 규칙에 따라 `k` 를 고르기

이제 정합니다. 우리 서비스의 기준은 이렇습니다.

> **답에 필요한 근거를 되도록 다 건지되(재현율 0.9 이상), 관련 없는 글은 되도록 적게 넣는다.**

- 이 기준을 만족하는 **가장 작은 K** 를 변수 **`chosen_k`** 에 **정수**로 담으세요.
- 왜 그 값인지 근거가 되는 두 수 — 그 K 의 재현율과 정밀도 — 를 각각 **`chosen_recall`**, **`chosen_precision`** 에 담으세요(실수).
- 세 값 모두 `bs_k_table` 에서 **찾아내야** 합니다. 눈으로 보고 손으로 적으면 안 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건을 만족하는 행만 남기고, 그중 K 가 가장 작은 행을 고른다.

세부구현:
1. 재현율이 기준 이상인 행만 걸러 낸다.
2. 남은 행을 K 오름차순으로 보고 첫 행을 고른다.
3. 그 행에서 K 와 두 지표를 꺼내 각각 담는다.
   3-1. K 는 정수여야 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(chosen_k, int), 'chosen_k 는 정수여야 합니다'
assert chosen_k == 2, '기준을 만족하는 가장 작은 K 를 다시 확인하세요'
# 손으로 적은 값이 아니라 bs_k_table 에서 꺼낸 값인지 대조한다
chosen_row = bs_k_table[bs_k_table['K'] == chosen_k].iloc[0]
assert abs(chosen_recall - chosen_row['R']) < 1e-9, 'chosen_recall 은 bs_k_table 에서 꺼내세요'
assert abs(chosen_precision - chosen_row['P']) < 1e-9, 'chosen_precision 은 bs_k_table 에서 꺼내세요'
assert chosen_recall >= 0.9, '고른 K 가 기준을 만족하지 않습니다'
# 더 작은 K 는 기준을 만족하지 않아야 한다(가장 작은 K 여야 하므로)
smaller_rows = bs_k_table[bs_k_table['K'] < chosen_k]
assert (smaller_rows['R'] < 0.9).all(), '더 작은 K 로도 기준을 만족합니다 - 다시 고르세요'
print('✅ 통과!')

### 3단계 — 못 찾은 문항 읽기 (서술형)

평균만 보고 끝내지 않습니다. **`chosen_k`** 로 쟀을 때 **`Hit` 이 0 인 문항**을 찾아 출력하세요 — 질문·정답 라벨·실제 검색 결과를 나란히 봅니다. 변수 이름은 **`missed`**(문항 id 의 리스트)로 하세요. 적중 여부는 1번에서 만든 **`hit_at_k`** 로 판단합니다(`0.0` 이면 못 찾은 문항입니다).

그 문항을 실제로 읽고, **아래 markdown 셀에** 두 가지를 적으세요.

1. 왜 못 찾았다고 생각하나요? (질문의 낱말과 정답 FAQ 의 낱말을 견주어 보세요)
2. `K` 를 더 키우면 이 문항이 해결될까요? 표를 근거로 답하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 고른 K 로 검색해 정답이 하나도 안 들어왔는지 본다.

세부구현:
1. 평가셋을 돌며 고른 K 로 검색한다.
2. 정답 목록과 겹치는 것이 하나도 없으면 그 문항 id 를 모은다.
3. 모은 문항의 질문·정답·검색 결과를 차례로 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(missed, list) and len(missed) == 1, '고른 K 에서 못 찾은 문항은 하나입니다'
# missed 가 손으로 적은 값이 아니라 실제로 재서 나온 값인지 대조한다
want_missed = [r.query_id for r in bs_eval.itertuples()
         if hit_at_k(bs_search_ids(r.query, chosen_k), r.gold_chunks.split('|'), chosen_k) == 0.0]
assert missed == want_missed, 'missed 는 실제로 재서 모은 목록이어야 합니다'
print('✅ 통과!')

**답안** *(아래에 두 물음의 답을 서술하세요)*

*(여기에 자신의 판단을 서술하세요)*

---
수고했어요! 검색을 재는 **눈금 네 개**를 손으로 만들고, 그것으로 서로 다른 두 검색기를 재어 **쓸 `k` 를 근거를 가지고** 골랐습니다. 이 과제에서는 모델을 한 번도 부르지 않았지요 — **판단의 근거를 수치로 만드는 일**에는 모델이 필요 없습니다. 평가셋을 **어떻게 만드는지**가 궁금하다면 `부록_평가셋_구축.ipynb` 를 보세요.